[070] 소스 코드 Ⅱ-28 데이터 읽기: https://github.com/seongui2030/Section2/blob/main/Titanic.csv

In [48]:
import os
import urllib.request

# 1. ./data 폴더 생성 (이미 존재하면 무시)
os.makedirs("./data", exist_ok=True)

# 2. milknew.csv 파일을 ./data/milknew.csv 경로로 다운로드
url = "https://raw.githubusercontent.com/seongui2030/Section2/main/Titanic.csv"
urllib.request.urlretrieve(url, "./data/Titanic.csv")
print("다운로드 완료!")

다운로드 완료!


In [49]:
#  교재 Ⅱ-01 데이터 읽기
import pandas as pd
import numpy as np
df = pd.read_csv('/content/data/Titanic.csv')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [50]:
#  교재 Ⅱ-02 데이터 속성 확인하기
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

[071] 데이터의 결측치 유무 확인하기

In [10]:
#  교재 Ⅱ-03 데이터 형태 확인하기
df.shape

(891, 12)

In [51]:
#  교재 Ⅱ-04 결측치 유무 확인하기
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [52]:
#  교재 Ⅱ-06 핵심 속성 추출
df_core_columns = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
df_core_columns.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked'],
      dtype='object')

In [53]:
# 교재 Ⅱ-05 이상치 유무 파악하기

import numpy as np

# 1. IQR을 이용한 'Fare(요금)' 속성의 이상치 개수 확인 함수 정의
def outliers_iqr(dt, col):
    q1, q3 = np.percentile(dt[col], [25, 75])
    iqr = q3 - q1
    outlier_top = q3 + (1.5 * iqr)
    outlier_bottom = q1 - (1.5 * iqr)

    outliers_1 = dt[col] > outlier_top
    outliers_2 = dt[col] < outlier_bottom
    return dt[outliers_1 | outliers_2]

# 'Fare' 속성의 이상치 확인
fare_outliers = outliers_iqr(df_core_columns, 'Fare')
print(f"Fare 속성의 이상치 개수: {len(fare_outliers)}")

# 2. 결측치(누락된 값) 유무 파악
print("\n[속성별 결측치 개수]")
print(df_core_columns.isnull().sum())

Fare 속성의 이상치 개수: 116

[속성별 결측치 개수]
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64


[075] 데이터 전처리하기

In [54]:
# 교재 Ⅱ-07 결측치 제거하기(속성 제거)
df_notnull = df_core_columns.drop(['Cabin'], axis =1)

In [55]:
# 교재 Ⅱ-08 결측치 제거하기(데이터 제거)
df_notnull = df_notnull.dropna()

In [56]:
# 교재 Ⅱ-09 이상치가 있는 데이터를 평균값으로 치환해 보자
# 'Fare' 속성의 이상치를 결측치로 변경하고 확인해 보자
find_outliers = outliers_iqr(df_notnull, 'Fare')
df_notnull.loc[find_outliers.index, 'Fare'] = np.nan
df_notnull.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,95
Embarked,0


In [57]:
# 교재 Ⅱ-10 'Fare' 속성의 평균값을 구하여, 결측치에 평균값을 입력하고 확인해 보자.
mean_Fare = df_notnull['Fare'].mean(axis = 0)
# 기존 코드 (경고 발생)
# df_notnull['Fare'].fillna(df_notnull['Fare'].mean(), inplace=True)

# 수정된 코드 (권장)
df_notnull['Fare'] = df_notnull['Fare'].fillna(df_notnull['Fare'].mean())
df_notnull['Fare']

,Fare
0,7.250000
1,19.026803
2,7.925000
3,53.100000
4,8.050000
...,...
885,29.125000
886,13.000000
887,30.000000
889,30.000000


In [58]:
# 교재 75, Ⅱ-11 데이터 저장하기
df_notnull.to_csv('./data/Titanic2.csv')

[094] 기계학습 알고리즘의 모델 구현하기

[095]  Ⅱ-28. 데이터 읽기

In [59]:
# Ⅱ-28. 데이터 읽기
df_notnull = pd.read_csv('/content/data/Titanic2.csv')

In [61]:
# Ⅱ-30. 데이터 확인하기
df_notnull.head()

,Unnamed: 0,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,0,3,male,22.0,1,0,7.250000,S
1,1,1,1,female,38.0,1,0,19.026803,C
2,2,1,3,female,26.0,0,0,7.925000,S
3,3,1,1,female,35.0,1,0,53.100000,S
4,4,0,3,male,35.0,0,0,8.050000,S


[095] 소스 코드 Ⅱ-30 데이터 전처리하기(데이터의 형태 변경)

In [62]:
o_sex = pd.get_dummies(df_notnull.Sex)
o_embarked = pd.get_dummies(df_notnull.Embarked, prefix = 'town')
df_notnull = pd.concat([df_notnull, o_sex, o_embarked], axis = 1)

[096] 소스 코드 Ⅱ-31 데이터 전처리하기(전처리한 데이터 확인)

In [63]:
df_notnull.head(3)

,Unnamed: 0,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,female,male,town_C,town_Q,town_S
0,0,0,3,male,22.0,1,0,7.250000,S,False,True,False,False,True
1,1,1,1,female,38.0,1,0,19.026803,C,True,False,True,False,False
2,2,1,3,female,26.0,0,0,7.925000,S,True,False,False,False,True


[096] 소스 코드 Ⅱ-32 데이터 전처리하기(불필요한 속성 제거)

In [64]:
df_notnull = df_notnull.drop(['Sex', 'Embarked'], axis = 1)
df_notnull.head(3)

,Unnamed: 0,Survived,Pclass,Age,SibSp,Parch,Fare,female,male,town_C,town_Q,town_S
0,0,0,3,22.0,1,0,7.250000,False,True,False,False,True
1,1,1,1,38.0,1,0,19.026803,True,False,True,False,False
2,2,1,3,26.0,0,0,7.925000,True,False,False,False,True


[097] 소스 코드 Ⅱ-33 데이터 전처리하기(데이터 표현 변경)

In [65]:
df_notnull['Fare'] = df_notnull['Fare'].round(2)
df_notnull.head(3)

,Unnamed: 0,Survived,Pclass,Age,SibSp,Parch,Fare,female,male,town_C,town_Q,town_S
0,0,0,3,22.0,1,0,7.25,False,True,False,False,True
1,1,1,1,38.0,1,0,19.03,True,False,True,False,False
2,2,1,3,26.0,0,0,7.92,True,False,False,False,True


[097] 소스 코드 Ⅱ-34 데이터 전처리하기(종속 변수와 독립 변수 지정)

In [66]:
X = df_notnull.loc[:, 'Pclass':]
y = df_notnull['Survived']

[097] 소스 코드 Ⅱ-35 데이터 전처리하기(데이터 분할과 형태 확인)

In [67]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y)
print('train data 개수:', X_train.shape)
print('test data 개수:', X_test.shape)

train data 개수: (534, 10)
test data 개수: (178, 10)


[098] 소스 코드 Ⅱ-36 모델 생성하기(K-최근접 이웃)

In [68]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors = 5)
knn.fit(X_train, y_train)

KNeighborsClassifier()

[098] 소스 코드 Ⅱ-37 모델의 성능 평가하기

In [69]:
from sklearn import metrics
y_pred = knn.predict(X_test)
acc = metrics.accuracy_score(y_test, y_pred)
acc

0.6797752808988764

[098] 소스 코드 Ⅱ-38 최적의 K 값 찾기

In [70]:
k = 10
acc_array = np.zeros(k)
for k in np.arange(1,k+1,1):
    knn = KNeighborsClassifier(n_neighbors = k).fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, y_pred)
    acc_array[k-1] = acc
    print('k: ', k, '정확도: ', acc)

k:  1 정확도:  0.6573033707865169
k:  2 정확도:  0.651685393258427
k:  3 정확도:  0.6910112359550562
k:  4 정확도:  0.6404494382022472
k:  5 정확도:  0.6797752808988764
k:  6 정확도:  0.651685393258427
k:  7 정확도:  0.6853932584269663
k:  8 정확도:  0.6404494382022472
k:  9 정확도:  0.6853932584269663
k:  10 정확도:  0.6235955056179775


[099] 소스 코드 Ⅱ-39 모델 활용하기

In [71]:
new_data1 = np.array([[3, 22.0, 1, 0, 52.1000, 0, 1, 1, 0, 0]])
new_data1 = pd.DataFrame(new_data1, columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'female', 'male', 'town_C', 'town_Q', 'town_S'])
prediction = knn.predict(new_data1)
print('생존 여부', prediction)

생존 여부 [1]


[099] 소스 코드 Ⅱ-40 모델 활용하기

In [72]:
new_data2 = np.array([[3, 55.0, 1, 0, 7.1000, 1, 0, 1, 0, 0]])
new_data2 = pd.DataFrame(new_data2, columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'female', 'male', 'town_C', 'town_Q', 'town_S'])
prediction = knn.predict(new_data2)
print('생존 여부', prediction)

생존 여부 [0]
